# **Project Name** - Global Terrorism Exploratory Data Analysis (1970–2017)


##### **Project Type** - EDA
##### **Contribution** - Individual
##### **Team Member -** Manan Bhatt

# **Project Summary -**

This project presents a comprehensive Exploratory Data Analysis (EDA) of the United Nations Global Terrorism Analysis (UNGTA) dataset, which records over 180,000 terrorist incidents from 1970 to 2017 across the globe.

The primary goal of this analysis is to uncover meaningful patterns in global terrorism — identifying which regions are most affected, which attack methods are most common, which groups are most active, and how the global terrorism landscape has shifted over nearly five decades.

**Dataset Overview:** The dataset contains detailed records of each attack including date, country, region, city, attack type, target type, weapon used, perpetrator group, number of casualties (killed and wounded), and whether the attack was successful.

**Key Steps Performed:**
1. **Data Loading & First Look:** Loaded the dataset and examined its shape, data types, and initial rows to understand the structure.
2. **Data Cleaning:** Identified and handled missing values using appropriate strategies — median imputation for numerical columns (nkill, nwound) and 'Unknown' fill for categorical columns. Outliers in casualty columns were capped using the IQR method.
3. **Feature Engineering:** Created new columns such as `total_casualties` (nkill + nwound) and `decade` to enable time-based trend analysis.
4. **Univariate Analysis:** Studied the distribution of individual features — yearly attack frequency, regional distribution, attack types, and weapon types.
5. **Bivariate Analysis:** Explored relationships between pairs of variables — casualties by region, success rates by attack type, top groups by attack count.
6. **Multivariate Analysis:** Created heatmaps and pair plots to understand complex interactions between multiple variables simultaneously.

**Key Findings:** Terrorism surged dramatically after 2001 and peaked around 2014–2015. The Middle East & North Africa, South Asia, and Sub-Saharan Africa are the most affected regions. Bombings/Explosions are the most frequent attack type. The Taliban, ISIL, and Shining Path are the most prolific groups. Private citizens and military personnel are the most frequently targeted groups.

**Business Value:** These insights are directly useful to governments, NGOs, security agencies, and policy analysts for resource allocation, risk assessment, early warning systems, and counter-terrorism strategy formulation.

# **GitHub Link -**

https://github.com/[your-username]/global-terrorism-eda

# **Problem Statement**


Terrorism is one of the most critical threats to global peace and security. Despite decades of counter-terrorism efforts, attacks have continued to rise in frequency and lethality in many parts of the world.

**The problem:** Security agencies, governments, and international organizations lack a data-driven, visual understanding of terrorism trends — including which regions are escalating, which attack methods cause maximum casualties, and which groups are most dangerous.

**This project aims to answer:**
- How has global terrorism evolved from 1970 to 2017?
- Which countries and regions are the hotspots?
- What attack types and weapons are most lethal?
- Which terrorist groups are most active and deadly?
- What patterns exist in timing, targets, and tactics?

#### **Define Your Business Objective?**

**Business Objective:** To analyze global terrorism data from 1970–2017 and extract actionable insights that help:
1. **Government & Security Agencies** — Identify high-risk regions and periods to allocate security resources efficiently.
2. **NGOs & Humanitarian Organizations** — Anticipate where civilian protection and aid are most needed.
3. **Policy Makers** — Design evidence-based counter-terrorism policies targeting the most lethal attack types and groups.
4. **Travel & Insurance Industry** — Assess regional risk levels for travelers and businesses.
5. **Researchers & Academics** — Provide a cleaned, well-visualized dataset for further study.

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# ── Standard Libraries ──────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')   # suppress non-critical warnings

# ── Data Manipulation ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Plot Style ────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print("✅ All libraries imported successfully.")

### Dataset Loading

In [ ]:
# Load the Global Terrorism Dataset
# NOTE: Update the path below if running locally (e.g., 'globalterrorismdb_0718dist.csv')
# On Google Colab, upload the file first using the Files panel on the left.

try:
    df = pd.read_csv('Global Terrorism Data.csv', encoding='latin-1', engine='python')
    print(f"✅ Dataset loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print("❌ File not found. Please upload the CSV file and update the path above.")


### Dataset First View

In [ ]:
# View first 5 rows to understand structure
df.head()

### Dataset Rows & Columns count

In [ ]:
# Check dataset dimensions
print(f"Number of Rows    : {df.shape[0]:,}")
print(f"Number of Columns : {df.shape[1]}")

### Dataset Information

In [ ]:
# Datatypes and non-null counts for all columns
df.info(verbose=True, show_counts=True)

### Dataset Columns

In [ ]:
# Print all column names
print("Columns in Dataset:")
for col in df.columns:
    print(f"  - {col}")

### Dataset Describe

In [ ]:
# Statistical summary of numerical columns
df.describe().T

### Check Unique Values in each column

In [ ]:
# Count unique values in each column
print("Unique value counts per column:")
print(df.nunique().to_string())

### Check Duplicates

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print(f"✅ Duplicates dropped. New shape: {df.shape}")
else:
    print("✅ No duplicate rows found.")

### Check Missing Values / Null Values

In [ ]:
# Calculate missing value percentages for each column
missing = pd.DataFrame({
    'Missing Count' : df.isnull().sum(),
    'Missing %'     : (df.isnull().sum() / len(df) * 100).round(2)
})
# Show only columns with at least 1 missing value, sorted descending
missing = missing[missing['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(f"Columns with missing values: {len(missing)}")
missing

## ***2. Understanding Your Variables***

In [ ]:
# ── Select the key columns we'll use for analysis ────────────────────────────
# The full dataset has 135 columns — we focus on the most analytically relevant ones.

KEY_COLS = [
    'iyear', 'imonth', 'iday',          # Date fields
    'country_txt', 'region_txt', 'city', # Location fields
    'latitude', 'longitude',             # Geo coordinates
    'attacktype1_txt',                   # Attack type
    'targtype1_txt',                     # Target type
    'gname',                             # Terrorist group name
    'weaptype1_txt',                     # Weapon type
    'nkill', 'nwound',                   # Casualties
    'success', 'suicide'                 # Attack outcome flags
]

# Keep only existing key columns (handles slight naming differences in dataset versions)
KEY_COLS = [c for c in KEY_COLS if c in df.columns]
df = df[KEY_COLS].copy()

print(f"✅ Working dataframe shape: {df.shape}")
df.head(3)

In [ ]:
# ── Variable description table ────────────────────────────────────────────────
var_desc = {
    'iyear'           : ('int',   'Year of the incident'),
    'imonth'          : ('int',   'Month of the incident'),
    'iday'            : ('int',   'Day of the incident'),
    'country_txt'     : ('str',   'Country where attack occurred'),
    'region_txt'      : ('str',   'World region of the attack'),
    'city'            : ('str',   'City of the attack'),
    'latitude'        : ('float', 'Latitude coordinate'),
    'longitude'       : ('float', 'Longitude coordinate'),
    'attacktype1_txt' : ('str',   'Primary attack method'),
    'targtype1_txt'   : ('str',   'Primary target type'),
    'gname'           : ('str',   'Name of terrorist/perpetrator group'),
    'weaptype1_txt'   : ('str',   'Primary weapon used'),
    'nkill'           : ('float', 'Number of people killed'),
    'nwound'          : ('float', 'Number of people wounded'),
    'success'         : ('int',   'Whether attack was successful (1=Yes, 0=No)'),
    'suicide'         : ('int',   'Whether attack was a suicide mission (1=Yes, 0=No)'),
}
desc_df = pd.DataFrame(var_desc, index=['Type', 'Description']).T
desc_df

## ***3. Data Wrangling***

### Data Wrangling Code

In [ ]:
# ── Step 1: Handle Missing Values ────────────────────────────────────────────

# Numerical columns: fill NaN with median (robust to outliers)
num_cols = ['nkill', 'nwound', 'latitude', 'longitude']
for col in num_cols:
    if col in df.columns:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"  '{col}' — NaN filled with median = {median_val:.2f}")

# Categorical columns: fill NaN with 'Unknown'
cat_cols = ['city', 'attacktype1_txt', 'targtype1_txt', 'gname', 'weaptype1_txt']
for col in cat_cols:
    if col in df.columns:
        n_missing = df[col].isna().sum()
        df[col] = df[col].fillna('Unknown')
        print(f"  '{col}' — {n_missing} NaN filled with 'Unknown'")

print(f"\n✅ After cleaning — remaining nulls: {df.isnull().sum().sum()}")

In [ ]:
# ── Step 2: Feature Engineering ───────────────────────────────────────────────

# Total casualties = killed + wounded
df['total_casualties'] = df['nkill'] + df['nwound']

# Decade — useful for trend analysis across time periods
df['decade'] = (df['iyear'] // 10) * 10
df['decade'] = df['decade'].astype(str) + 's'

# Lethality flag — attacks that killed at least 1 person
df['is_lethal'] = (df['nkill'] >= 1).astype(int)

print("✅ Feature engineering complete.")
print(f"   New columns: total_casualties, decade, is_lethal")
df[['nkill', 'nwound', 'total_casualties', 'decade', 'is_lethal']].head()

In [ ]:
# ── Step 3: Outlier Treatment (IQR Capping) ───────────────────────────────────
# We use IQR capping (Winsorization) instead of dropping outliers
# because extreme events (mass casualty attacks) are real and meaningful.
# Capping preserves the record while reducing distortion in visualizations.

def cap_outliers_iqr(series, lower_pct=0.01, upper_pct=0.99):
    """Cap outliers at specified percentiles."""
    lower = series.quantile(lower_pct)
    upper = series.quantile(upper_pct)
    return series.clip(lower=lower, upper=upper)

for col in ['nkill', 'nwound', 'total_casualties']:
    before_max = df[col].max()
    df[col] = cap_outliers_iqr(df[col])
    after_max = df[col].max()
    print(f"  '{col}' max: {before_max:.0f} → {after_max:.0f} (after capping)")

print("\n✅ Outlier capping complete using 1st–99th percentile range.")

In [ ]:
# ── Final clean dataset overview ───────────────────────────────────────────────
print(f"Final dataset shape : {df.shape}")
print(f"Year range          : {df['iyear'].min()} – {df['iyear'].max()}")
print(f"Countries covered   : {df['country_txt'].nunique()}")
print(f"Unique groups       : {df['gname'].nunique()}")
print(f"Total killed (capped): {df['nkill'].sum():,.0f}")
print(f"Total wounded (capped): {df['nwound'].sum():,.0f}")

## ***4. Data Vizualization, Storytelling & Experimenting with charts***

### ── UNIVARIATE ANALYSIS ──────────────────────────────────────────────

#### Chart - 1 : Terrorist Attacks Per Year (1970–2017)

In [ ]:
# Chart 1: Line chart — Attacks per year
yearly = df.groupby('iyear').size().reset_index(name='attacks')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(yearly['iyear'], yearly['attacks'], color='steelblue', linewidth=2, marker='o', markersize=3)
ax.fill_between(yearly['iyear'], yearly['attacks'], alpha=0.15, color='steelblue')
ax.axvline(x=2001, color='red', linestyle='--', linewidth=1.2, label='9/11 (2001)')
ax.axvline(x=2014, color='orange', linestyle='--', linewidth=1.2, label='ISIL peak (2014)')
ax.set_title('Global Terrorist Attacks Per Year (1970–2017)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Attacks')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
A **line chart with area fill** is ideal for displaying a continuous time-series variable (attack frequency over 47 years). It clearly shows trends, acceleration, and decline over time.

##### 2. What insights were found?
- Attacks were relatively low and stable from 1970–1997.
- A sharp upward trend began after **2001 (post-9/11)**, reflecting increased global terrorist activity.
- The peak occurred around **2014–2015**, largely driven by the rise of ISIL in Iraq and Syria.
- Post-2015, there is a declining trend, possibly due to intensified counter-terrorism operations.

##### 3. Business Impact?
**Positive:** Security agencies can use this trend to justify resource scale-up during peak periods and design cyclical preparedness plans. The post-2015 decline can validate certain policy interventions.
**Negative:** The sharp 2001–2015 growth implies that early warning systems were insufficient — governments need predictive analytics, not just reactive responses.

#### Chart - 2 : Top 15 Countries by Number of Attacks

In [ ]:
# Chart 2: Horizontal bar — top 15 most attacked countries
top_countries = df['country_txt'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(x=top_countries.values, y=top_countries.index, palette='Reds_r', ax=ax)
ax.set_title('Top 15 Countries by Number of Terrorist Attacks', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Attacks')
ax.set_ylabel('Country')
for i, v in enumerate(top_countries.values):
    ax.text(v + 100, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
A **horizontal bar chart** is ideal for ranking categorical variables with long labels (country names). The horizontal layout makes country names readable without rotation.

##### 2. What insights were found?
- **Iraq** tops the list by a significant margin, followed by **Pakistan** and **Afghanistan**.
- These three countries collectively account for a disproportionately large share of global attacks.
- **India**, **Colombia**, and **Philippines** also appear — indicating terrorism is not limited to the Middle East.

##### 3. Business Impact?
**Positive:** Enables targeted diplomatic, military, and humanitarian resource allocation to the highest-impact nations.
**Negative:** Concentration of attacks in a few countries may lead to under-attention to rising threats in other regions.

#### Chart - 3 : Distribution of Attack Types

In [ ]:
# Chart 3: Bar chart — attack type distribution
attack_types = df['attacktype1_txt'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=attack_types.values, y=attack_types.index, palette='coolwarm_r', ax=ax)
ax.set_title('Distribution of Attack Types', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Attacks')
ax.set_ylabel('Attack Type')
for i, v in enumerate(attack_types.values):
    ax.text(v + 200, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
A bar chart effectively displays the frequency distribution of a categorical variable (attack type), making it easy to rank methods by usage.

##### 2. What insights were found?
- **Bombing/Explosion** is by far the most common attack type, accounting for nearly half of all attacks.
- **Armed Assault** is second, followed by **Assassination**.
- High-casualty methods like bombings indicate that mass-casualty intent is prevalent.

##### 3. Business Impact?
**Positive:** Security forces can prioritize bomb detection technologies and trained squads in high-risk areas.
**Negative:** The prevalence of explosives-based attacks indicates that IED (Improvised Explosive Device) proliferation is a major ongoing threat that requires international arms control measures.

#### Chart - 4 : Distribution of Target Types

In [ ]:
# Chart 4: Bar chart — who are the targets?
target_types = df['targtype1_txt'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=target_types.values, y=target_types.index, palette='OrRd_r', ax=ax)
ax.set_title('Top 12 Target Types in Terrorist Attacks', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Attacks')
ax.set_ylabel('Target Type')
for i, v in enumerate(target_types.values):
    ax.text(v + 100, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
Understanding who is targeted is as critical as understanding how. A ranked bar chart clearly highlights the most vulnerable groups.

##### 2. What insights were found?
- **Private Citizens & Property** is the most frequent target — terrorists deliberately target civilians to create fear.
- **Military** and **Police** are next, indicating attacks aimed at undermining state authority.
- **Government** and **Business** targets reflect political and economic disruption motives.

##### 3. Business Impact?
**Positive:** Civilian-protection protocols and early-warning systems in public spaces should be the highest priority for governments.
**Negative:** The heavy targeting of police and military suggests morale and manpower impact on security forces, which can degrade state response capacity over time.

#### Chart - 5 : Attacks by World Region

In [ ]:
# Chart 5: Pie chart — regional distribution of attacks
region_counts = df['region_txt'].value_counts()

fig, ax = plt.subplots(figsize=(9, 9))
wedges, texts, autotexts = ax.pie(
    region_counts.values,
    labels=region_counts.index,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.8,
    colors=sns.color_palette('tab10', len(region_counts))
)
for at in autotexts:
    at.set_fontsize(8)
ax.set_title('Distribution of Attacks by World Region', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
A pie chart shows proportional distribution across categories. It quickly communicates which regions dominate global terrorism at a glance.

##### 2. What insights were found?
- **Middle East & North Africa** and **South Asia** together account for approximately 50% of all global attacks.
- **Sub-Saharan Africa** is a rising region of concern.
- **Western Europe** and **North America**, despite high media coverage, represent a smaller proportion of total attacks.

##### 3. Business Impact?
**Positive:** International aid organizations and peacekeepers can prioritize funding for Middle East, South Asia, and Sub-Saharan Africa.
**Negative:** The Western world may over-invest in domestic security while underfunding prevention in the most-affected regions.

### ── BIVARIATE ANALYSIS ──────────────────────────────────────────────

#### Chart - 6 : Attacks per Year by Region (Line Chart — Num vs Time)

In [ ]:
# Chart 6: Multi-line chart — top 5 regions' attack trends over years
top5_regions = df['region_txt'].value_counts().head(5).index
region_year = df[df['region_txt'].isin(top5_regions)].groupby(['iyear', 'region_txt']).size().reset_index(name='attacks')

fig, ax = plt.subplots(figsize=(14, 6))
for region in top5_regions:
    data = region_year[region_year['region_txt'] == region]
    ax.plot(data['iyear'], data['attacks'], marker='o', markersize=2, linewidth=1.5, label=region)

ax.set_title('Attacks Over Time — Top 5 Regions', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Attacks')
ax.legend(title='Region', fontsize=8)
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
A multi-line chart enables comparison of temporal trends across multiple categories simultaneously — revealing which regions are escalating, stable, or declining.

##### 2. What insights were found?
- **Middle East & North Africa** saw the sharpest spike post-2011 (Arab Spring / ISIL).
- **South Asia** has maintained a consistently high and growing rate since the 1990s.
- **Sub-Saharan Africa** has grown considerably since 2010, driven by groups like Boko Haram.
- **Western Europe** and **South America** trends are relatively flat.

##### 3. Business Impact?
**Positive:** Time-based regional intelligence allows pre-emptive resource deployment before peaks.
**Negative:** The sharp spike in the Middle East post-2011 shows that political instability (Arab Spring) directly fuels terrorism — a warning for regions undergoing political transitions.

#### Chart - 7 : Average Casualties by Attack Type (Numerical–Categorical)

In [ ]:
# Chart 7: Bar chart — mean total casualties by attack type
casualty_by_attack = df.groupby('attacktype1_txt')['total_casualties'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=casualty_by_attack.values, y=casualty_by_attack.index, palette='magma_r', ax=ax)
ax.set_title('Average Total Casualties by Attack Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Average Casualties (Killed + Wounded)')
ax.set_ylabel('Attack Type')
for i, v in enumerate(casualty_by_attack.values):
    ax.text(v + 0.1, i, f'{v:.1f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
Counting attacks (frequency) tells only part of the story. Average casualties per attack type reveals **lethality** — which is more important for impact assessment.

##### 2. What insights were found?
- **Chemical** and **Biological** attacks have very high average casualties despite being rare — extremely dangerous per event.
- **Hostage Taking** and **Hijacking** also result in high average casualties.
- While **Bombing** is the most frequent, its average casualties per attack are moderate — many bombs are small IEDs.
- **Unarmed Assault** and **Facility/Infrastructure Attack** have lower casualty averages.

##### 3. Business Impact?
**Positive:** Even rare attack types (chemical/bio) need dedicated rapid-response teams due to mass casualty potential.
**Negative:** Focusing only on frequency (bombings) while ignoring lethality-per-event could lead to underpreparedness for unconventional attacks.

#### Chart - 8 : Top 15 Terrorist Groups by Number of Attacks (Categorical–Categorical context)

In [ ]:
# Chart 8: Horizontal bar — top 15 most active groups (excluding 'Unknown')
top_groups = df[df['gname'] != 'Unknown']['gname'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(x=top_groups.values, y=top_groups.index, palette='viridis_r', ax=ax)
ax.set_title('Top 15 Most Active Terrorist Groups', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Attacks')
ax.set_ylabel('Group Name')
for i, v in enumerate(top_groups.values):
    ax.text(v + 20, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
Identifying the most prolific groups is critical for counter-terrorism strategy. A horizontal ranked bar shows clear group-level volume at a glance.

##### 2. What insights were found?
- **Taliban** leads by a large margin, followed by **ISIL** and **Shining Path (Peru)**.
- Groups like **Al-Shabaab**, **Boko Haram**, and **IRA** represent diverse regional threats.
- A large proportion of attacks are attributed to 'Unknown' groups — indicating intelligence gaps.

##### 3. Business Impact?
**Positive:** Counter-terrorism resources can be specifically targeted at the top 5 groups responsible for the majority of attacks.
**Negative:** The high 'Unknown' proportion highlights the need for better intelligence-gathering infrastructure.

#### Chart - 9 : Monthly Distribution of Attacks (Seasonality)

In [ ]:
# Chart 9: Bar chart — attacks by month (excluding month 0 — data quality)
monthly = df[df['imonth'] > 0]['imonth'].value_counts().sort_index()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(month_names, monthly.values, color=sns.color_palette('Blues_d', 12))
ax.set_title('Monthly Distribution of Terrorist Attacks', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Attacks')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
Seasonality analysis reveals if terrorism follows any time-of-year patterns, which is critical for security planning and resource scheduling.

##### 2. What insights were found?
- Attacks are broadly distributed throughout the year but show a slight increase in **May–July** and **October–December**.
- No single month dominates, but the summer and year-end periods see marginally higher activity.
- This may be linked to political cycles (elections, religious events, anniversaries) in high-attack regions.

##### 3. Business Impact?
**Positive:** Security agencies can increase alert levels and resource deployment during historically higher-risk months.
**Negative:** Near-uniform monthly distribution means year-round vigilance is required — no 'safe months' exist.

#### Chart - 10 : Attack Success Rate by Attack Type (Numerical–Categorical)

In [ ]:
# Chart 10: Bar chart — success rate (%) by attack type
success_rate = df.groupby('attacktype1_txt')['success'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=success_rate.values, y=success_rate.index, palette='RdYlGn_r', ax=ax)
ax.set_title('Attack Success Rate (%) by Attack Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Success Rate (%)')
ax.set_ylabel('Attack Type')
ax.axvline(x=50, color='black', linestyle='--', linewidth=1, label='50% line')
for i, v in enumerate(success_rate.values):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
Success rate per attack type reveals the **effectiveness** of each method, which is valuable for understanding how defenses can be improved.

##### 2. What insights were found?
- Most attack types have success rates above 80%, meaning terrorists generally achieve their tactical objectives.
- **Unarmed Assault** and **Assassination** have the highest success rates.
- **Hijacking** has a lower success rate — possibly due to improved aviation security post-9/11.
- This indicates that prevention must be the primary strategy, since once an attack begins, it usually succeeds.

##### 3. Business Impact?
**Positive:** High success rates should push governments to invest more in prevention and early detection rather than response.
**Negative:** The near-universal high success rate signals that current deterrence mechanisms are often insufficient to stop attacks in progress.

#### Chart - 11 : Total Casualties by Region (Numerical–Categorical)

In [ ]:
# Chart 11: Bar chart — total casualties (killed + wounded) by region
region_casualties = df.groupby('region_txt')['total_casualties'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=region_casualties.values, y=region_casualties.index, palette='Reds_r', ax=ax)
ax.set_title('Total Casualties (Killed + Wounded) by Region', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Casualties')
ax.set_ylabel('Region')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
Frequency of attacks and human cost are different metrics. A region can have fewer attacks but devastating casualties. This chart separates frequency from impact.

##### 2. What insights were found?
- **Middle East & North Africa** suffers the highest total casualties — by a large margin.
- **South Asia** is second, followed by **Sub-Saharan Africa**.
- **Western Europe** and **North America** have relatively low total casualties despite high media attention.

##### 3. Business Impact?
**Positive:** Humanitarian aid, medical response resources, and trauma support should be concentrated in the Middle East and South Asia.
**Negative:** The disparity between media coverage (Western Europe/North America) and actual casualties (Middle East/South Asia) can lead to misallocated international empathy and funding.

#### Chart - 12 : Killed vs. Wounded — Scatter Plot (Numerical–Numerical)

In [ ]:
# Chart 12: Scatter plot — nkill vs nwound colored by region
# Sample 5000 for readability
sample = df.sample(5000, random_state=42)

fig, ax = plt.subplots(figsize=(10, 7))
regions_list = sample['region_txt'].unique()
palette = sns.color_palette('tab10', len(regions_list))
for i, region in enumerate(regions_list):
    subset = sample[sample['region_txt'] == region]
    ax.scatter(subset['nkill'], subset['nwound'], alpha=0.4, s=10,
               color=palette[i], label=region)

ax.set_title('Killed vs. Wounded per Attack (colored by Region)', fontsize=13, fontweight='bold')
ax.set_xlabel('Number Killed (nkill)')
ax.set_ylabel('Number Wounded (nwound)')
ax.legend(fontsize=7, title='Region', markerscale=2)
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
A scatter plot is the standard tool for exploring the relationship between two continuous numerical variables. Adding region as color adds a third dimension.

##### 2. What insights were found?
- Most attacks cause low casualties (clustered near origin), with a long tail of high-impact attacks.
- There is a **positive correlation** between killed and wounded — more lethal attacks also wound more people.
- Middle East & North Africa attacks (orange) show the widest spread across both axes, reflecting high-severity attacks.
- Many attacks kill but injure few, and vice versa, suggesting different weapon types and tactics.

##### 3. Business Impact?
**Positive:** High-killed/low-wounded events indicate precision weapons (assassinations); high-wounded/low-killed suggests blast weapons (bombs in open areas). This insight shapes first-responder training.
**Negative:** The clustering near zero means most attacks cause little harm individually, but cumulative impact over thousands of events is enormous.

#### Chart - 13 : Suicide Attacks Over the Years

In [ ]:
# Chart 13: Stacked bar or line — suicide vs non-suicide attacks by year
suicide_yearly = df.groupby(['iyear', 'suicide']).size().unstack(fill_value=0)
suicide_yearly.columns = ['Non-Suicide', 'Suicide']

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(suicide_yearly.index, suicide_yearly['Non-Suicide'], label='Non-Suicide', color='steelblue')
ax.bar(suicide_yearly.index, suicide_yearly['Suicide'], bottom=suicide_yearly['Non-Suicide'], label='Suicide', color='crimson')
ax.set_title('Suicide vs. Non-Suicide Attacks by Year', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Attacks')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
A stacked bar chart shows both the total volume and the composition of suicide vs. non-suicide attacks across time, revealing the trend in the most extreme form of commitment by attackers.

##### 2. What insights were found?
- Suicide attacks were nearly non-existent before the mid-1990s.
- They rose sharply post-2001 and again after 2011, correlating with ISIL's rise.
- Suicide attacks, while fewer in total count, tend to be far more lethal and harder to prevent.
- The increasing proportion of suicide attacks signals a growing ideological commitment among certain groups.

##### 3. Business Impact?
**Positive:** The post-2015 leveling off of suicide attacks may reflect successful counter-recruitment and de-radicalization programs.
**Negative:** Even a small number of suicide attacks causes disproportionate casualties, requiring specialized security protocols (body scanners, behavioral profiling) at public venues.

#### Chart - 14 : Top 10 Groups by Total Casualties Caused

In [ ]:
# Chart 14: Bar chart — top 10 groups by total casualties
group_casualties = (
    df[df['gname'] != 'Unknown']
    .groupby('gname')['total_casualties']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=group_casualties.values, y=group_casualties.index, palette='rocket_r', ax=ax)
ax.set_title('Top 10 Terrorist Groups by Total Casualties Caused', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Casualties (Killed + Wounded)')
ax.set_ylabel('Group Name')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(group_casualties.values):
    ax.text(v + 50, i, f'{v:,.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
Attack frequency and total casualties paint very different pictures of a group's danger. This chart focuses on **impact** — which groups cause the most human harm.

##### 2. What insights were found?
- **ISIL** tops the casualty chart despite entering the dataset only around 2013 — reflecting extreme lethality per attack.
- The **Taliban** is second — consistent with their long history and geographic reach.
- **Boko Haram** ranks high in casualties despite being a regional African group — reflecting their use of mass-casualty bombings and kidnappings.

##### 3. Business Impact?
**Positive:** International targeted sanctions, asset freezes, and military pressure should prioritize these top-10 groups.
**Negative:** ISIL's rapid rise and extreme lethality within just a few years shows how quickly new groups can become globally dangerous, requiring faster intelligence adaptation.

#### Chart - 15 : Heatmap — Attacks by Region and Attack Type (Categorical–Categorical)

In [ ]:
# Chart 15: Heatmap — count of attacks by region vs attack type
pivot = df.pivot_table(index='region_txt', columns='attacktype1_txt', aggfunc='size', fill_value=0)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.3, ax=ax,
            annot_kws={'size': 7})
ax.set_title('Heatmap: Attack Count by Region and Attack Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Attack Type')
ax.set_ylabel('Region')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
A heatmap efficiently encodes two categorical dimensions with a numerical value (count). It reveals which region–attack-type combinations are most common at a single glance.

##### 2. What insights were found?
- **Middle East & North Africa + Bombing/Explosion** is the hottest cell — by far the most common combination.
- **South Asia + Armed Assault** is also a high-frequency cell.
- **Hostage Taking** is most prevalent in South America and South Asia.
- Western regions show relatively low intensity across all attack types.

##### 3. Business Impact?
**Positive:** Security equipment procurement (e.g., bomb-detection vs. hostage-negotiation teams) can be regionally customized based on dominant attack types.
**Negative:** The dominance of a single cell (Middle East + Bombing) could cause tunnel-vision in security planning, ignoring other dangerous combinations.

### ── MULTIVARIATE ANALYSIS ──────────────────────────────────────────────

#### Chart - 16 : Decade-wise Attacks by Region (Grouped Bar Chart)

In [ ]:
# Chart 16: Grouped bar — attacks per decade for top 6 regions
top6_regions = df['region_txt'].value_counts().head(6).index
decade_region = (
    df[df['region_txt'].isin(top6_regions)]
    .groupby(['decade', 'region_txt'])
    .size()
    .reset_index(name='attacks')
)

fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(data=decade_region, x='decade', y='attacks', hue='region_txt',
            palette='tab10', ax=ax)
ax.set_title('Attacks per Decade by Region (Top 6 Regions)', fontsize=13, fontweight='bold')
ax.set_xlabel('Decade')
ax.set_ylabel('Number of Attacks')
ax.legend(title='Region', fontsize=8, loc='upper left')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
A grouped bar chart shows how each region evolved decade by decade, enabling comparative historical analysis across multiple regions simultaneously.

##### 2. What insights were found?
- South America dominated in the 1970s–1990s (Cold War-era groups like Shining Path, FARC).
- The Middle East surged dramatically in the 2010s.
- Sub-Saharan Africa's 2010s attacks surpassed all its previous decades combined.
- South Asia has shown consistent growth across all decades.

##### 3. Business Impact?
**Positive:** Historical decade trends help predict future at-risk regions, enabling proactive diplomatic engagement and capacity-building.
**Negative:** The dramatic shift from South America to Middle East/Africa shows geopolitical changes can rapidly redirect terrorism — requiring adaptive intelligence frameworks.

#### Chart - 17 : Top 10 Cities Most Targeted by Terrorist Attacks

In [ ]:
# Chart 17: Bar chart — top 10 most attacked cities (excluding 'Unknown')
top_cities = df[df['city'].str.lower() != 'unknown']['city'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=top_cities.values, y=top_cities.index, palette='cubehelix_r', ax=ax)
ax.set_title('Top 10 Most Attacked Cities', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Attacks')
ax.set_ylabel('City')
for i, v in enumerate(top_cities.values):
    ax.text(v + 10, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
Cities are the operational theaters of terrorism. This chart narrows from country to city-level analysis for more granular security planning.

##### 2. What insights were found?
- **Baghdad** (Iraq) is the most attacked city in the world — by a wide margin.
- **Karachi** and **Belfast** also appear prominently.
- Most top-10 cities are conflict-zone capitals, confirming that urban centers in unstable regions are primary targets.

##### 3. Business Impact?
**Positive:** City-level data enables precision resource deployment — security upgrades, surveillance infrastructure, and emergency services can be concentrated in highest-risk cities.
**Negative:** Capital cities are symbolic targets. Attacks in these cities carry disproportionate psychological and political impact beyond the physical damage.

#### Chart - 18 : Lethality Rate by Region (% of Attacks that Killed ≥1 person)

In [ ]:
# Chart 18: Bar chart — lethality rate by region
lethality = df.groupby('region_txt')['is_lethal'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=lethality.values, y=lethality.index, palette='flare', ax=ax)
ax.set_title('Lethality Rate by Region (% of Attacks with ≥1 Death)', fontsize=13, fontweight='bold')
ax.set_xlabel('Lethality Rate (%)')
ax.set_ylabel('Region')
ax.axvline(x=50, color='black', linestyle='--', linewidth=1)
for i, v in enumerate(lethality.values):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
Lethality rate (proportion of attacks that kill) separates high-frequency regions from high-severity regions — crucial for impact-weighted analysis.

##### 2. What insights were found?
- **Sub-Saharan Africa** has the highest lethality rate — most attacks there result in deaths.
- **Middle East & North Africa** is also above 50%.
- **Western Europe** has a lower lethality rate — suggesting many attacks are disrupted before causing deaths (e.g., failed bombings, police intervention).
- This shows that attack **prevention** is more effective in developed regions.

##### 3. Business Impact?
**Positive:** Regions with high lethality need faster emergency medical response and trauma infrastructure investment.
**Negative:** The lower lethality in Western Europe vs. higher frequency elsewhere suggests that wealth and infrastructure correlate with survival, creating a global equity gap in terrorism outcomes.

#### Chart - 19 : Weapon Type vs. Average Casualties (Multivariate — box-like analysis)

In [ ]:
# Chart 19: Box plot — casualties distribution by weapon type
# Exclude 'Unknown' and very rare types
top_weapons = df[df['weaptype1_txt'] != 'Unknown']['weaptype1_txt'].value_counts().head(8).index
weapon_data = df[df['weaptype1_txt'].isin(top_weapons)]

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=weapon_data, x='total_casualties', y='weaptype1_txt',
            palette='Set2', fliersize=2, ax=ax)
ax.set_title('Casualty Distribution by Weapon Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Casualties (Killed + Wounded)')
ax.set_ylabel('Weapon Type')
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
A box plot shows the distribution, spread, and outliers of a numerical variable across categories — much more informative than just mean bars for comparing weapon lethality.

##### 2. What insights were found?
- **Explosives/Bombs** have the widest casualty spread and highest outliers — capable of causing mass casualties.
- **Firearms** show a consistent median casualty level across attacks.
- **Incendiary** weapons have lower median casualties but occasional extreme outliers (mass arson events).
- **Melee** weapons (knives etc.) produce the lowest casualties on average, though the psychological impact is high.

##### 3. Business Impact?
**Positive:** Controls on explosive precursor chemicals are the single highest-impact policy intervention based on this data.
**Negative:** The wide variance for explosives means both small IEDs and mass-casualty bombs fall under the same category — response planning needs to cover both scenarios.

#### Chart - 20 : Correlation Heatmap of Numerical Variables

In [ ]:
# Chart 20: Correlation heatmap — numerical columns
num_df = df[['iyear', 'imonth', 'nkill', 'nwound', 'total_casualties',
             'success', 'suicide', 'is_lethal']].copy()

corr = num_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True, ax=ax)
ax.set_title('Correlation Heatmap of Numerical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
A correlation heatmap reveals linear relationships between all numerical features simultaneously. It is a foundational step for any EDA to understand which variables are related.

##### 2. What insights were found?
- **nkill and nwound** show a moderate positive correlation — more lethal attacks also wound more people.
- **total_casualties** is strongly correlated with both nkill and nwound (as expected, since it's their sum).
- **suicide** and **is_lethal** show a positive correlation — suicide attacks are more likely to kill.
- **iyear** has a slight positive correlation with total_casualties — attacks have become more lethal over time.
- **imonth** shows very low correlations with everything — no strong seasonal pattern in lethality.

#### Chart - 21 : Pair Plot — Key Numerical Features colored by Lethality

In [ ]:
# Chart 21: Pair plot — sample for performance
pair_sample = df[['nkill', 'nwound', 'total_casualties', 'success', 'is_lethal']].sample(3000, random_state=42)

g = sns.pairplot(pair_sample, hue='is_lethal', palette={0: 'steelblue', 1: 'crimson'},
                 diag_kind='kde', plot_kws={'alpha': 0.3, 's': 10},
                 vars=['nkill', 'nwound', 'total_casualties'])
g.fig.suptitle('Pair Plot: Key Numerical Features (Red = Lethal, Blue = Non-lethal)', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

##### 1. Why did you pick this chart?
A pair plot provides a comprehensive overview of pairwise relationships among multiple numerical variables at once. Using hue (lethal vs. non-lethal) adds a classification dimension.

##### 2. What insights were found?
- **Lethal attacks (red)** are clearly separated from **non-lethal (blue)** in all scatter plots — indicating that nkill, nwound, and total_casualties are good differentiators of attack severity.
- The diagonal KDE plots show lethal attacks have a heavier right tail — a small number cause extremely high casualties.
- Most non-lethal attacks cluster tightly near zero on all axes, while lethal ones are more spread out.

## **5. Solution to Business Objective**

#### What do you suggest the client to achieve Business Objective?

Based on our comprehensive EDA of 47 years of global terrorism data, here are evidence-based recommendations:

**1. Regional Resource Allocation:**
The UN and governments should concentrate counter-terrorism resources in **Middle East & North Africa**, **South Asia**, and **Sub-Saharan Africa** — which together account for ~65% of all attacks and the majority of casualties.

**2. Attack-Type Specific Countermeasures:**
Since **Bombing/Explosion** is the most common attack type, the highest ROI security investment is in **explosive precursor chemical controls**, **IED detection technology**, and **bomb-disposal capacity** in high-risk regions.

**3. Group-Targeted Counter-Terrorism:**
Taliban, ISIL, and Al-Shabaab collectively cause the most deaths. Focused intelligence operations, asset freezes, and military pressure on these top-5 groups would yield the greatest impact reduction.

**4. Civilian Protection Priority:**
Private citizens are the #1 target. Investment in **community early-warning systems**, **public space security**, and **community de-radicalization programs** should be prioritized.

**5. Predictive Analytics:**
The post-2001 surge and the post-2015 decline suggest that certain geopolitical events trigger terrorism waves. Governments should invest in **predictive modeling** that incorporates political instability indicators.

**6. Intelligence Gap Closure:**
A significant proportion of attacks are attributed to 'Unknown' groups. Strengthening **human intelligence (HUMINT)** networks and **data-sharing between national agencies** is essential.

**7. Suicide Attack Prevention:**
Given that suicide attacks are more lethal, counter-radicalization programs — particularly **online de-radicalization** — should be scaled up in regions where ISIL and Taliban recruit.

# **Conclusion**

This EDA of the UNGTA Global Terrorism Dataset (1970–2017) revealed several critical patterns:

- Global terrorism has shown a dramatic **upward trajectory since 2001**, peaking in 2014–2015 before declining slightly.
- **Iraq, Pakistan, and Afghanistan** are the world's most affected countries, and **Baghdad** is the most attacked city.
- **Bombing/Explosion** is the dominant attack type, but **chemical and biological attacks** cause the highest casualties per event.
- **Private citizens** are the primary target, followed by military and police.
- **Taliban and ISIL** are the most active and most lethal groups respectively.
- **Suicide attacks**, while fewer in number, are significantly more lethal and have been rising since the mid-1990s.
- **Middle East & North Africa** suffered the most attacks and casualties in the 2010s.
- Sub-Saharan Africa is a **rapidly rising** terrorism hotspot that requires immediate international attention.

The findings from this analysis can directly support evidence-based policy-making, humanitarian planning, and counter-terrorism strategy at both national and international levels.

### ***Hurrah! You have successfully completed your EDA Capstone Project !!!***